## ERP115712

**paper:** [PMID: 32050897](https://pmc.ncbi.nlm.nih.gov/articles/PMC7014947/) - Pan-tissue transcriptome analysis of long noncoding RNAs in the American beaver Castor canadensis, 2020

**date, curator:** 2026-04-08, Sara Carsanaro

**notes**
* this experiment is related to ERP105716

### annotation summary

In [22]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,skeletal muscle tissue,UBERON:0001134,skeletal muscle tissue,perfect match
1,lung,UBERON:0002048,lung,perfect match
2,liver,UBERON:0002107,liver,perfect match
3,kidney,UBERON:0000082,adult mammalian kidney,perfect match
4,intestine,UBERON:0000160,intestine,perfect match
5,heart,UBERON:0000948,heart,perfect match
6,castor sac,UBERON:0036019,castor sac,perfect match
7,brain,UBERON:0000955,brain,perfect match
8,tongue,UBERON:0001723,tongue,perfect match
9,toe webbing,UBERON:0008439,webbed interdigital region between pedal digits,perfect match


In [23]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,adult,UBERON:0018241,prime adult stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "ERP115712"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
E-MTAB-8038
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,,,,skeletal muscle tissue,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 9,SAMEA5703355,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 9,,,,adult,public,TRANSCRIPTOMIC,RANDOM
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,,,,lung,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 8,SAMEA5703354,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 8,,,,adult,public,TRANSCRIPTOMIC,RANDOM
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,,,,liver,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 7,SAMEA5703353,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 7,,,,adult,public,TRANSCRIPTOMIC,RANDOM
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,,,,kidney,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 6,SAMEA5703352,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 6,,,,adult,public,TRANSCRIPTOMIC,RANDOM
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,,,,intestine,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 5,SAMEA5703351,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 5,,,,adult,public,TRANSCRIPTOMIC,RANDOM
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,,,,heart,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 4,SAMEA5703350,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 4,,,,adult,public,TRANSCRIPTOMIC,RANDOM
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,,,,castor sac,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 3,SAMEA5703349,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 3,,,,adult,public,TRANSCRIPTOMIC,RANDOM
7,ERX3390658,ERP115712,Illumina HiSeq 3000,ERS3507005,UBERON:0000955,brain,,,,brain,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 2,SAMEA5703348,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 2,,,,adult,public,TRANSCRIPTOMIC,RANDOM
8,ERX3390657,ERP115712,Illumina HiSeq 3000,ERS3507004,UBERON:0001723,tongue,,,,tongue,adult,perfect match,not documented,,F,,,51338,,,,,,Sample 16,SAMEA5703347,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalo

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['blood' 'brain' 'castor sac' 'heart' 'intestine' 'kidney' 'liver' 'lung'
 'mature ovarian follicle' 'placenta' 'skeletal muscle tissue' 'spleen'
 'stomach' 'tail' 'toe webbing' 'tongue']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['adult']


In [7]:
# all
library.loc[:,'stageId'] = 'UBERON:0018241'
library.loc[:,'stageName'] = 'prime adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'perfect match'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 9,SAMEA5703355,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 9,,,,adult,public,TRANSCRIPTOMIC,RANDOM
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 8,SAMEA5703354,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 8,,,,adult,public,TRANSCRIPTOMIC,RANDOM
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 7,SAMEA5703353,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 7,,,,adult,public,TRANSCRIPTOMIC,RANDOM
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 6,SAMEA5703352,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 6,,,,adult,public,TRANSCRIPTOMIC,RANDOM
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 5,SAMEA5703351,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 5,,,,adult,public,TRANSCRIPTOMIC,RANDOM
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,UBERON:0018241,prime adult stage,,heart,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 4,SAMEA5703350,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 4,,,,adult,public,TRANSCRIPTOMIC,RANDOM
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,UBERON:0018241,prime adult stage,,castor sac,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 3,SAMEA5703349,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 3,,,,adult,public,TRANSCRIPTOMIC,RANDOM
7,ERX3390658,ERP115712,Illumina HiSeq 3000,ERS3507005,UBERON:0000955,brain,UBERON:0018241,prime adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 2,SAMEA5703348,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [8]:
library.loc[:,'sex'] = 'F'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 9,SAMEA5703355,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 9,,,,adult,public,TRANSCRIPTOMIC,RANDOM
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 8,SAMEA5703354,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 8,,,,adult,public,TRANSCRIPTOMIC,RANDOM
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 7,SAMEA5703353,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 7,,,,adult,public,TRANSCRIPTOMIC,RANDOM
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 6,SAMEA5703352,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 6,,,,adult,public,TRANSCRIPTOMIC,RANDOM
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 5,SAMEA5703351,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 5,,,,adult,public,TRANSCRIPTOMIC,RANDOM
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,UBERON:0018241,prime adult stage,,heart,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 4,SAMEA5703350,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 4,,,,adult,public,TRANSCRIPTOMIC,RANDOM
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,UBERON:0018241,prime adult stage,,castor sac,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 3,SAMEA5703349,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 3,,,,adult,public,TRANSCRIPTOMIC,RANDOM
7,ERX3390658,ERP115712,Illumina HiSeq 3000,ERS3507005,UBERON:0000955,brain,UBERON:0018241,prime adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,51338,,,,,,Sample 2,SAMEA5703348,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 9,SAMEA5703355,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 9,,,,adult,public,TRANSCRIPTOMIC,RANDOM
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 8,SAMEA5703354,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 8,,,,adult,public,TRANSCRIPTOMIC,RANDOM
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 7,SAMEA5703353,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 7,,,,adult,public,TRANSCRIPTOMIC,RANDOM
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 6,SAMEA5703352,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 6,,,,adult,public,TRANSCRIPTOMIC,RANDOM
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 5,SAMEA5703351,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 5,,,,adult,public,TRANSCRIPTOMIC,RANDOM
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,UBERON:0018241,prime adult stage,,heart,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 4,SAMEA5703350,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 4,,,,adult,public,TRANSCRIPTOMIC,RANDOM
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,UBERON:0018241,prime adult stage,,castor sac,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 3,SAMEA5703349,,,,,,,,,,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 3,,,,adult,public,TRANSCRIPTOMIC,RANDOM
7,ERX3390658,ERP115712,Illumina HiSeq 3000,ERS3507005,UBERON:0000955,brain,UBERON:0018241,prime ad

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [11]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
library.loc[:,'physiologicalStatus'] = 'pregnant'

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 9,SAMEA5703355,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 9,,,,adult,public,TRANSCRIPTOMIC,RANDOM
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 8,SAMEA5703354,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 8,,,,adult,public,TRANSCRIPTOMIC,RANDOM
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 7,SAMEA5703353,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 7,,,,adult,public,TRANSCRIPTOMIC,RANDOM
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 6,SAMEA5703352,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 6,,,,adult,public,TRANSCRIPTOMIC,RANDOM
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 5,SAMEA5703351,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 5,,,,adult,public,TRANSCRIPTOMIC,RANDOM
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,UBERON:0018241,prime adult stage,,heart,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 4,SAMEA5703350,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 4,,,,adult,public,TRANSCRIPTOMIC,RANDOM
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,UBERON:0018241,prime adult stage,,castor sac,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 3,SAMEA5703349,,,,,,,,,pregnant,,08/04/2026,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 3,,,,adult,public,TRANSCRIPTOMIC,RANDOM
7,ERX3390658,ERP115712,Illumina HiSeq 3000

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [12]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-04-08'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 9,SAMEA5703355,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 9,,,,adult,public,TRANSCRIPTOMIC,RANDOM
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 8,SAMEA5703354,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 8,,,,adult,public,TRANSCRIPTOMIC,RANDOM
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 7,SAMEA5703353,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 7,,,,adult,public,TRANSCRIPTOMIC,RANDOM
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 6,SAMEA5703352,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 6,,,,adult,public,TRANSCRIPTOMIC,RANDOM
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 5,SAMEA5703351,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 5,,,,adult,public,TRANSCRIPTOMIC,RANDOM
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,UBERON:0018241,prime adult stage,,heart,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 4,SAMEA5703350,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 4,,,,adult,public,TRANSCRIPTOMIC,RANDOM
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,UBERON:0018241,prime adult stage,,castor sac,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 3,SAMEA5703349,,,,,,,,,pregnant,SAC,2026-04-08,Tissue samples were collected from a trapped beaver Zymo Direct-zol™ RNA MiniPrep Catalog Number R2050 Wafergen PrepX PolyA mRNA Isolation Kit #400047,,E-MTAB-8038:Sample 3,,,,adult,public,TRANSCRIPTOMIC,RANDOM
7,ERX3390658,ERP11571

#### comments

In [13]:
library.loc[:,'comment'] = 'PMID: 32050897'

#### save complete file with correct columns

In [14]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 9,SAMEA5703355,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
1,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 8,SAMEA5703354,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
2,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 7,SAMEA5703353,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
3,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 6,SAMEA5703352,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
4,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 5,SAMEA5703351,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
5,ERX3390660,ERP115712,Illumina HiSeq 3000,ERS3507007,UBERON:0000948,heart,UBERON:0018241,prime adult stage,,heart,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 4,SAMEA5703350,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
6,ERX3390659,ERP115712,Illumina HiSeq 3000,ERS3507006,UBERON:0036019,castor sac,UBERON:0018241,prime adult stage,,castor sac,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 3,SAMEA5703349,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
7,ERX3390658,ERP115712,Illumina HiSeq 3000,ERS3507005,UBERON:0000955,brain,UBERON:0018241,prime adult stage,,brain,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 2,SAMEA5703348,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
8,ERX3390657,ERP115712,Illumina HiSeq 3000,ERS3507004,UBERON:0001723,tongue,UBERON:0018241,prime adult stage,,tongue,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 16,SAMEA5703347,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
9,ERX3390656,ERP115712,Illumina HiSeq 3000,ERS3507003,UBERON:0008439,webbed interdigital region between pedal digits,UBERON:0018241,prime adult stage,,toe webbing,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 15,SAMEA5703346,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08


### experiment annotations

In [15]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115712,RNAseq of 16 North American Beaver (Castor canadensis) tissues,RNAseq from the following tissue samples: skeletal muscle; kidney; spleen; ovaries; placenta; castor gland; tail; toe webbing; whole blood; brain; lung; liver; heart; stomach; tongue; intestine,SRA,,,,,,,PRJEB32966,,,,,


#### experiment and protocol details

In [16]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

16

In [17]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115712,RNAseq of 16 North American Beaver (Castor canadensis) tissues,RNAseq from the following tissue samples: skeletal muscle; kidney; spleen; ovaries; placenta; castor gland; tail; toe webbing; whole blood; brain; lung; liver; heart; stomach; tongue; intestine,SRA,total,Bgee 1K,16,TruSeq Stranded mRNA,full_length,,PRJEB32966,,,,,


#### paper and xrefs

In [18]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '32050897'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC7014947/'
experiment.loc[:,'DOI'] = '10.1186/s12864-019-6432-4'
experiment.loc[:,'xrefs'] = 'E-MTAB-8038[ArrayExpress]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,ERP115712,RNAseq of 16 North American Beaver (Castor canadensis) tissues,RNAseq from the following tissue samples: skeletal muscle; kidney; spleen; ovaries; placenta; castor gland; tail; toe webbing; whole blood; brain; lung; liver; heart; stomach; tongue; intestine,SRA,total,Bgee 1K,16,TruSeq Stranded mRNA,full_length,,PRJEB32966,32050897,https://pmc.ncbi.nlm.nih.gov/articles/PMC7014947/,10.1186/s12864-019-6432-4,E-MTAB-8038[ArrayExpress],


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [19]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [20]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [21]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [24]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [25]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
59031,SRX18043724,SRP404822,MGISEQ-2000RS,SRS15550318,UBERON:0000178,blood,UBERON:0007222,late adult stage,,Blood,30,perfect match,not documented,missing child term,F,,,9646,,,polyA,,,Model organism or animal sample from Ailuropod...,SAMN31476541,30,year,,,,,PMID: 36548828,,,SAC,2026-04-08
59032,ERX2273482,ERP105716,Illumina MiSeq,ERS2047307,UBERON:0000468,multicellular organism,UBERON:0018241,prime adult stage,,mixed pool 16 tissues,adult,other,not documented,perfect match,F,,,51338,PrepX RNA-Seq,full_length,polyA,,,Sample 1,SAMEA104429365,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
59033,ERX3390665,ERP115712,Illumina HiSeq 3000,ERS3507012,UBERON:0001134,skeletal muscle tissue,UBERON:0018241,prime adult stage,,skeletal muscle tissue,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 9,SAMEA5703355,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
59034,ERX3390664,ERP115712,Illumina HiSeq 3000,ERS3507011,UBERON:0002048,lung,UBERON:0018241,prime adult stage,,lung,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 8,SAMEA5703354,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
59035,ERX3390663,ERP115712,Illumina HiSeq 3000,ERS3507010,UBERON:0002107,liver,UBERON:0018241,prime adult stage,,liver,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 7,SAMEA5703353,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
59036,ERX3390662,ERP115712,Illumina HiSeq 3000,ERS3507009,UBERON:0000082,adult mammalian kidney,UBERON:0018241,prime adult stage,,kidney,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 6,SAMEA5703352,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08
59037,ERX3390661,ERP115712,Illumina HiSeq 3000,ERS3507008,UBERON:0000160,intestine,UBERON:0018241,prime adult stage,,intestine,adult,perfect match,not documented,perfect match,F,,,51338,TruSeq Stranded mRNA,full_length,polyA,,,Sample 5,SAMEA5703351,,,,,,,PMID: 32050897,,pregnant,SAC,2026-04-08


In [26]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1156,SRP404822,Ailuropoda melanoleuca Transcriptome or Gene e...,Analysis and comparison of transcriptome infor...,SRA,total,Bgee 1K,48,,,,PRJNA893394,36548828,https://pmc.ncbi.nlm.nih.gov/articles/PMC9784451/,10.3390/vetsci9120667,,
1157,ERP105716,North American Beaver (Castor canadensis) RNAs...,RNAseq from Illumina MiSeq 75bp paired end for...,SRA,total,Bgee 1K,1,PrepX RNA-Seq,full_length,,PRJEB23937,32050897,https://pmc.ncbi.nlm.nih.gov/articles/PMC7014947/,10.1186/s12864-019-6432-4,E-MTAB-6258[ArrayExpress],
1158,ERP115712,RNAseq of 16 North American Beaver (Castor can...,RNAseq from the following tissue samples: skel...,SRA,total,Bgee 1K,16,TruSeq Stranded mRNA,full_length,,PRJEB32966,32050897,https://pmc.ncbi.nlm.nih.gov/articles/PMC7014947/,10.1186/s12864-019-6432-4,E-MTAB-8038[ArrayExpress],


### add annotations to git

In [27]:
! git pull

Already up to date.


In [28]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [ ]:
! git status

In [29]:
! git add $git_experiment_path $git_library_path

In [30]:
! git commit -m $commit_message_exp

[develop 8ca9d54] adding annotated bulk experiment ERP115712
 2 files changed, 17 insertions(+)


In [31]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.29 KiB | 2.29 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   10b2270..8ca9d54  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push